In [ ]:
import glob
import os
import re
import numpy as np
import pandas as pd

# Garante a pasta de saída para o dataset consolidado
os.makedirs("dados", exist_ok=True)


def padronizar_nome_colunas(df):
    """Remove caracteres especiais, acentos e espaços das colunas para busca segura."""
    cols = []
    for c in df.columns:
        c_str = str(c).lower().strip()
        c_str = re.sub(r"[áàãâ]", "a", c_str)
        c_str = re.sub(r"[éèê]", "e", c_str)
        c_str = re.sub(r"[íì]", "i", c_str)
        c_str = re.sub(r"[óòõô]", "o", c_str)
        c_str = re.sub(r"[úù]", "u", c_str)
        c_str = re.sub(r"[ç]", "c", c_str)
        c_str = re.sub(r"[^a-z0-9_]", "_", c_str)
        c_str = re.sub(r"_+", "_", c_str).strip("_")
        cols.append(c_str)
    df.columns = cols
    return df


def identificar_coluna_chave(df):
    """Identifica dinamicamente a coluna de id (CNPJ ou Nome da Instituição)."""
    # Prioridade 1: CNPJ
    for c in df.columns:
        if "cnpj" in c:
            return c
    # Prioridade 2: Variações de Nome/Instituição
    for c in df.columns:
        if any(term in c for term in ["instituic", "nome", "inst"]):
            return c
    # Fallback: Segunda coluna (padrão do BCB caso a primeira seja o Ranking)
    return df.columns[1] if len(df.columns) > 1 else df.columns[0]


def processar_painel_trimestral(pasta_origem="dados_bcb_olinda"):
    print(f"🚀 Lendo dados diretamente de '{pasta_origem}'...")

    arquivos = glob.glob(os.path.join(pasta_origem, "IFDATA_*_*.csv"))
    trimestres = sorted(
        list(
            set(
                [
                    re.search(r"(\d{6})", f).group(1)
                    for f in arquivos
                    if re.search(r"(\d{6})", f)
                ]
            )
        )
    )

    if not trimestres:
        print(f"⚠️ Nenhum arquivo no padrão IFDATA_YYYYMM_Relatorio.csv encontrado em '{pasta_origem}'.")
        return

    print(f"🔍 Encontrados {len(trimestres)} períodos/trimestres para consolidar.")

    lista_painel = []

    for tri in trimestres:
        print(f"⚡ Processando Período: {tri}...")

        relatorios_tri = {}
        for rel in ["Resumo", "Ativo", "Passivo", "Resultado"]:
            p = os.path.join(pasta_origem, f"IFDATA_{tri}_{rel}.csv")
            if os.path.exists(p):
                df_temp = pd.read_csv(p, sep=";", encoding="utf-8-sig", low_memory=False)
                df_temp = padronizar_nome_colunas(df_temp)
                relatorios_tri[rel] = df_temp

        if "Resumo" not in relatorios_tri:
            continue

        df_base = relatorios_tri["Resumo"].copy()

        # Merge horizontal flexível
        for rel_nome, df_rel in relatorios_tri.items():
            if rel_nome == "Resumo":
                continue

            # Identifica as chaves de cada tabela dinamicamente
            chave_base = identificar_coluna_chave(df_base)
            chave_rel = identificar_coluna_chave(df_rel)

            # Evita duplicar colunas que já existem na base (mantendo apenas a chave de junção)
            cols_novas = [c for c in df_rel.columns if c not in df_base.columns or c == chave_rel]

            if chave_base == chave_rel:
                df_base = pd.merge(
                    df_base,
                    df_rel[cols_novas],
                    on=chave_base,
                    how="left",
                    suffixes=("", f"_{rel_nome.lower()}"),
                )
            else:
                df_base = pd.merge(
                    df_base,
                    df_rel[cols_novas],
                    left_on=chave_base,
                    right_on=chave_rel,
                    how="left",
                    suffixes=("", f"_{rel_nome.lower()}"),
                )

        df_base["trimestre"] = tri
        df_base["ano"] = int(tri[:4])
        df_base["mes"] = int(tri[4:])
        lista_painel.append(df_base)

    df_completo = pd.concat(lista_painel, ignore_index=True)

    print("\n🧮 Calculando Indicadores CAMELS e Basileia...")

    def to_num(serie):
        if serie is None or serie.name not in df_completo.columns:
            return pd.Series(0.0, index=df_completo.index)
        return pd.to_numeric(
            serie.astype(str).str.replace(".", "", regex=False).str.replace(",", ".", regex=False),
            errors="coerce",
        ).fillna(0.0)

    def buscar_coluna(palavras_chave):
        for col in df_completo.columns:
            if all(kw in col for kw in palavras_chave):
                return df_completo[col]
        return pd.Series(0.0, index=df_completo.index)

    # Extração das contas contábeis
    ativo_total = to_num(buscar_coluna(["ativo", "total"]))
    patrimonio_liquido = to_num(buscar_coluna(["patrimonio", "liquido"]))
    lucro_liquido = to_num(buscar_coluna(["lucro"]))
    pdd = to_num(buscar_coluna(["provisao", "devedores"]))
    carteira_credito = to_num(buscar_coluna(["carteira", "credito"]))
    disponibilidades = to_num(buscar_coluna(["disponibilidade"]))
    capitacoes = to_num(buscar_coluna(["capitacao"]))

    # 1. Solvência / Basileia Proxy
    df_completo["camels_capital_alavancagem"] = np.where(
        ativo_total > 0, patrimonio_liquido / ativo_total, 0
    )

    # 2. Asset Quality (PDD / Carteira)
    df_completo["camels_pdd_carteira"] = np.where(
        carteira_credito > 0, np.abs(pdd) / carteira_credito, 0
    )

    # 3. Earnings (ROA e ROE)
    df_completo["camels_roa"] = np.where(ativo_total > 0, lucro_liquido / ativo_total, 0)
    df_completo["camels_roe"] = np.where(
        patrimonio_liquido > 0, lucro_liquido / patrimonio_liquido, 0
    )

    # 4. Liquidity
    df_completo["camels_liquidez"] = np.where(
        capitacoes > 0, disponibilidades / capitacoes, 0
    )

    # Marcador Target (1 = Insolvência/Risco Crítico, 0 = Saudável)
    df_completo["target"] = np.where(
        (df_completo["camels_capital_alavancagem"] <= 0) | (df_completo["camels_roe"] < -0.30),
        1,
        0,
    )

    # Salvando a base final
    arquivo_saida = "dados/dataset_painel_camels.csv"
    df_completo.to_csv(arquivo_saida, index=False, sep=";", encoding="utf-8-sig")

    print("=" * 60)
    print("✅ PROCESSAMENTO CONCLUÍDO COM SUCESSO!")
    print(f"📊 Dataset final salvo em: {arquivo_saida}")
    print(f"📈 Total de Balancetes: {len(df_completo)}")
    print(f"🚨 Instituições em Situação Crítica (Target=1): {df_completo['target'].sum()}")
    print("=" * 60)


if __name__ == "__main__":
    processar_painel_trimestral()

🚀 Lendo dados diretamente de 'dados_bcb_olinda'...
🔍 Encontrados 111 períodos/trimestres para consolidar.
⚡ Processando Período: 199412...
⚡ Processando Período: 199506...
⚡ Processando Período: 199512...
⚡ Processando Período: 199606...
⚡ Processando Período: 199612...
⚡ Processando Período: 199706...
⚡ Processando Período: 199712...
⚡ Processando Período: 199806...
⚡ Processando Período: 199812...
⚡ Processando Período: 199906...
⚡ Processando Período: 199912...
⚡ Processando Período: 200003...
